DSC540-T301 <br>
Term Project: Milestone 3<br>
Jeremy Barton <br>

# Milestone 3 - Cleaning/Formatting Website Data

## Extraction

### HTML Extraction of Monthly Treasury Data

The goal of this section is to extract fiscal spending data from the Monthly Treasury Statement (MTS) at <a href="https://fiscaldata.treasury.gov/datasets/monthly-treasury-statement/summary-of-receipts-outlays-and-the-deficit-surplus-of-the-u-s-government">FiscalData.Treasury.Gov</a>  with BeautifulSoup, then saving it to a dataframe.

The direction of this exercise follows Chapter 5 of "The Data Wrangling Workship" second edition on the "Introduction to Beautiful Soup 4 and Web Page Parsing". 

Due to the nature of data tables in the wild, there are several classes that separate us from the data, unlike book's example. Therefore slight changes such as using the `.get_text()` method have been added for efficiency.

In [1]:
from bs4 import BeautifulSoup

In [2]:
# Open the file
with open("/home/jeremy/Documents/GitHub/DSC540/TermProject/data/Monthly Treasury Statement.html") as mts:
    soup = BeautifulSoup(mts)
    print(type(soup))

<class 'bs4.BeautifulSoup'>


You can pass a file handler directly to the constructor of the BeautifulSoup
object and it will read the contents from the file that the handler is attached to.

Ensure consistent indentation with .prettyify from BeautifulSoup.

In [3]:
# print(soup.prettify)

Dereference the `<p>` tag as a mamaber variable of the BeautifuilSoup.

In [4]:
print(soup.p)

<p>The Monthly Treasury Statement (MTS) dataset provides information on the flow of money into and out of the U.S. Department of the Treasury. It includes how deficits are funded, such as borrowing from the public or reducing operating cash, and how surpluses are distributed. Further tables categorize spending (outlays) by department and agency, revenue (receipts) by specific taxes or other government sources of income, and transactions with trust funds such as Social Security or Medicare. All values are reported in millions of U.S. dollars.</p>


This finds just the first `<p>` tag, but to find them all, `findall` is used.

In [5]:
all_ps = soup.find_all('p')
print(all_ps)

[<p>The Monthly Treasury Statement (MTS) dataset provides information on the flow of money into and out of the U.S. Department of the Treasury. It includes how deficits are funded, such as borrowing from the public or reducing operating cash, and how surpluses are distributed. Further tables categorize spending (outlays) by department and agency, revenue (receipts) by specific taxes or other government sources of income, and transactions with trust funds such as Social Security or Medicare. All values are reported in millions of U.S. dollars.</p>, <p>The Monthly Treasury Statement (MTS) dataset provides information on the flow of money into and out of the U.S. Department of the Treasury. It includes how deficits are funded, such as borrowing from the public or reducing operating cash, and how surpluses are distributed. Further tables categorize spending (outlays) by department and agency, revenue (receipts) by specific taxes or other government sources of income, and transactions with 

Up until now only specific tags have been used to extrct data from webpages. BeautifulSoup also allows us to extract tags of the same type. Below is an example of using the `table` method to pull tags related to tables on the web. `<tr>`, `<td>`, `<th>`...

In [6]:
table = soup.table
print(table.get_text(separator="\n", strip=True))

# .get_text() extracts all non-tag text from the HTML element
    #   It allows us to add separators that will print
    #       the text in a far more readable format.

Name
Date
Size
Monthly Treasury State
ment.pdf
December 2025
3 MB
Download
Monthly Receipts, Outlays, and Deficit or Sur
plus.pdf
December 2025
104 KB
Download
Monthly Receipts, Outlays, and Deficit or Sur
plus.xls
December 2025
92 KB
Download
Distributed Offsetting Receipts by Department Re
port.pdf
December 2025
164 KB
Download
Distributed Offsetting Receipts by Department Report
(.xlsx)
December 2025
282 KB
Download
Budget Summary with Adjusted Differences Re
port.pdf
December 2025
701 KB
Download


The above data is only the immediate table found in the HTML. It is the table that contains the downloads for PDF files of the data.

Browsing all the possible children of a tag, and not only the immediate one. This is done using the `decendants` generator form bs4

In [7]:
children = table.children
des = table.descendants
print(len(list(children)), len(list(des)))

2 130


The above is the difference between children and descendents.

In the first attempt to extract data from the data table, only the first table was returned. This is class being accessed contains multiple tables. Inspecting the page can help identify the right parts of a webpage to pull the correct information. This is a more direct approach, as there are significantly more descendants the Treasury Data than the book's example.

Tables can be held in `classes` which can be accessed to find data tables.

Now we will use the .select() method from `bs4` to search inside a DIV for tables.

In [8]:
# Inside the .select() is where the div we want to search
#   for is placed.
tables = soup.select(
    "div.data-table-module--overlayContainerNoFooter--a46b8 table"
)

table = []
# Save output for use in next step
# Creating a loop to pick up all tables in the DIV
for table in tables:
    
     table.append(table.prettify())

# Making "table" equal to tables[1] will extract the
#   uneeded download file table.
table = tables[0]
# print(table)


In [9]:
# Check the new table data
# print(table.get_text(separator="\n", strip=True))

There are two tables in the `<div>` we're looking at. The trasury data is in the first of the two, so output should = `output[0]`, since `table[0]` is the true data we're searching for. The above was an operation to check where the true data is. The other table are just file download links.

### Saving to a DataFrame

Install pandas

In [10]:
import pandas as pd

Separate headers from the data.

In [11]:
rows = table.find_all('tr')[1:]
# Correct: headers is a <tr> tag
headers = table.find('tr')  # do NOT call getText() here
col_headers = [th.getText() for th in headers.findAll('th')]

col_headers

/tmp/ipykernel_11067/1751528950.py:4: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  col_headers = [th.getText() for th in headers.findAll('th')]


['Record Datemm/dd/yyyy - mm/dd/yyyy',
 'Parent ID\u200b',
 'Classification ID\u200b',
 'Classification Description\u200b',
 'Current Month Gross Receipts Amount\u200b',
 'Current Month Gross Outlay Amount\u200b',
 'Current Month Deficit Surplus Amount\u200b',
 'Table Number\u200b',
 'Source Line Number\u200b',
 'Print Order Sequence Number\u200b',
 'Line Code Number\u200b',
 'Data Type Code\u200b',
 'Record Type Code\u200b',
 'Sequence Level Number\u200b',
 'Sequence Number Code\u200b',
 'Fiscal Year\u200b',
 'Fiscal Quarter Number\u200b',
 'Calendar Year\u200b',
 'Calendar Quarter Number\u200b',
 'Calendar Month Number\u200b',
 'Calendar Day Number\u200b']

In [12]:
data_rows = []

# Clean row data to only include text inside 'td' or 'th' tags.
for row in rows:
    
    cells = [cell.get_text(strip=True) for cell in row.find_all(['td', 'th'])]
    data_rows.append(cells)

# data_rows

Carefully place the row data into a pandas dataframe.

In [13]:
treasury_data = [[td.getText() for td in tr.findAll('td')] \
            for tr in rows]
# df_data

/tmp/ipykernel_11067/2943969321.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  treasury_data = [[td.getText() for td in tr.findAll('td')] \


In [14]:
treasury_df = pd.DataFrame(treasury_data, columns=col_headers)

# strip whitespace from headers
treasury_df.columns = treasury_df.columns.str.strip()

treasury_df.head()


,Record Datemm/dd/yyyy - mm/dd/yyyy,Parent ID​,Classification ID​,Classification Description​,Current Month Gross Receipts Amount​,Current Month Gross Outlay Amount​,Current Month Deficit Surplus Amount​,Table Number​,Source Line Number​,Print Order Sequence Number​,...,Data Type Code​,Record Type Code​,Sequence Level Number​,Sequence Number Code​,Fiscal Year​,Fiscal Quarter Number​,Calendar Year​,Calendar Quarter Number​,Calendar Month Number​,Calendar Day Number​
0,12/31/2025,,57633926,FY 2025,,,,1,1,1,...,S,SL,1,1,2026,1,2025,4,12,31
1,12/31/2025,57633926,57633931,October,"$326,770,236,058.10","$584,220,273,025.31","$257,450,036,967.21",1,2,2,...,D,MTH,2,1.1,2026,1,2025,4,12,31
2,12/31/2025,57633926,57633935,November,"$301,754,299,812.79","$668,517,135,060.47","$366,762,835,247.68",1,3,3,...,D,MTH,2,1.2,2026,1,2025,4,12,31
3,12/31/2025,57633926,57633940,December,"$454,414,708,491.47","$541,146,327,504.76","$86,731,619,013.29",1,4,4,...,D,MTH,2,1.3,2026,1,2025,4,12,31
4,12/31/2025,57633926,57633942,January,"$513,294,342,651.93","$641,934,549,325.42","$128,640,206,673.49",1,5,5,...,D,MTH,2,1.4,2026,1,2025,4,12,31


## Cleaning

Transformations will be made to the web data.

### 1. Date Formatting

A few formatting issues should be addressed. There are mismatching date formats in the the treasury and wage dataframes. The ideal date format for `pandas` is `YYYY-MM-DD` (ISO 8601). Any dates should be converted to this format if they are not.

Date formatting will be done by detecting the values of `datetime64` fields in each dataframe.

From each dataframe, these are how the date formats look.

<b>`treasury_df`</b>

In [15]:
col = 'Record Datemm/dd/yyyy - mm/dd/yyyy'

# Check treasury_df dtype
print("Current dtype:", treasury_df[col].dtype)

print("Sample values:")
print(treasury_df[col].head())


Current dtype: str
Sample values:
0    12/31/2025
1    12/31/2025
2    12/31/2025
3    12/31/2025
4    12/31/2025
Name: Record Datemm/dd/yyyy - mm/dd/yyyy, dtype: str


The `datetime` conversion will be executed using the `pd.to_datetime()` method.

In [16]:
treasury_df[col] = pd.to_datetime(
    treasury_df[col],
    format='%m/%d/%Y',
    errors='raise'
)


In [17]:
# Check treasury_df dtype
print("Current dtype:", treasury_df[col].dtype)

print("Sample values:")
print(treasury_df[col].head())


Current dtype: datetime64[us]
Sample values:
0   2025-12-31
1   2025-12-31
2   2025-12-31
3   2025-12-31
4   2025-12-31
Name: Record Datemm/dd/yyyy - mm/dd/yyyy, dtype: datetime64[us]


While we're here, let's clean up the header name as well. It should be just "Record Date".

In [18]:
treasury_df.rename(
    columns={'Record Datemm/dd/yyyy - mm/dd/yyyy': 'Record Date'},
    inplace=True
)

treasury_df['Record Date'].dtype

dtype('<M8[us]')

`treasury_df` date formats have been converted.

### 2. Column Filtering

Much of the data that was pulled back from it's raw source, we won't need. Filtering the data will help keep only what is needed for the analysis.

To start, anything that isnt a month of the year in `Classification` should be removed. Fiscal year and Year-To-Date options exist for this column as well, which will without a doubt skew the data. 

In [19]:
# Check unique rows in Classification column
treasury_df['Classification Description​'].unique()

<StringArray>
[     'FY 2025',      'October',     'November',     'December',
      'January',     'February',        'March',        'April',
          'May',         'June',         'July',       'August',
    'September', 'Year-to-Date',      'FY 2026',      'FY 2024']
Length: 16, dtype: str

In [20]:
# Define valid months
valid_months = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

Remove non-month rows in `Classification`.

In [21]:
# Keep only rows where Classification Description is a valid month
treasury_df = treasury_df[treasury_df['Classification Description​'].isin(valid_months)]

# Drop any NaNs if needed
treasury_df = treasury_df.dropna(subset=['Classification Description​'])

In [22]:
treasury_df['Classification Description​'].unique()


<StringArray>
[  'October',  'November',  'December',   'January',  'February',     'March',
     'April',       'May',      'June',      'July',    'August', 'September']
Length: 12, dtype: str

Each of these months are attached to a dollar amount.

Now we'll begin filtering out unneeded fields.

<b>`treasury_df`</b>

In [23]:
treasury_df.columns.tolist()

['Record Date',
 'Parent ID\u200b',
 'Classification ID\u200b',
 'Classification Description\u200b',
 'Current Month Gross Receipts Amount\u200b',
 'Current Month Gross Outlay Amount\u200b',
 'Current Month Deficit Surplus Amount\u200b',
 'Table Number\u200b',
 'Source Line Number\u200b',
 'Print Order Sequence Number\u200b',
 'Line Code Number\u200b',
 'Data Type Code\u200b',
 'Record Type Code\u200b',
 'Sequence Level Number\u200b',
 'Sequence Number Code\u200b',
 'Fiscal Year\u200b',
 'Fiscal Quarter Number\u200b',
 'Calendar Year\u200b',
 'Calendar Quarter Number\u200b',
 'Calendar Month Number\u200b',
 'Calendar Day Number\u200b']

Another problem is that there are invisible characters in this data. Using `.replace()` will fix this data.

In [24]:
treasury_df.columns = [col.replace('\u200b', '') for col in treasury_df.columns]

For treasury_df retained columns will be `Record Date`, `Classification ID`, `Classification Description`, `Current Month Gross Receipts Amount`, `Current Month Gross Outlay Amount`, `Current Month Deficit Surplus Amount`.

In [25]:
t_retained = [
    'Record Date',                        # Date of the treasury record
    'Classification ID',                  # Unique numeric ID for the classification
    'Classification Description',         # Description of the classification (e.g., month/category)
    'Current Month Gross Receipts Amount',# Total receipts for the current month
    'Current Month Gross Outlay Amount',  # Total outlays/expenditures for the current month
    'Current Month Deficit Surplus Amount'# Net deficit or surplus for the current month
]

treasury_df = treasury_df[t_retained]


In [26]:
treasury_df.columns.tolist()

['Record Date',
 'Classification ID',
 'Classification Description',
 'Current Month Gross Receipts Amount',
 'Current Month Gross Outlay Amount',
 'Current Month Deficit Surplus Amount']

The treasury data came from the HTML file, so it's not expected that dtypes are applied other than the datetimes we just processed.

In [27]:
treasury_df['Record Date'] = pd.to_datetime(treasury_df['Record Date'], errors='coerce')
treasury_df['Classification ID'] = pd.to_numeric(treasury_df['Classification ID'], errors='coerce')
treasury_df['Classification Description'] = treasury_df['Classification Description'].astype(str)
treasury_df['Current Month Gross Receipts Amount'] = pd.to_numeric(treasury_df['Current Month Gross Receipts Amount'].astype(str).str.replace(',', '').str.replace('$',''), errors='coerce')
treasury_df['Current Month Gross Outlay Amount'] = pd.to_numeric(treasury_df['Current Month Gross Outlay Amount'].astype(str).str.replace(',', '').str.replace('$',''), errors='coerce')
treasury_df['Current Month Deficit Surplus Amount'] = pd.to_numeric(treasury_df['Current Month Deficit Surplus Amount'].astype(str).str.replace(',', '').str.replace('$',''), errors='coerce')

Now checking the datypes, they should no longer all be `str`.

In [28]:
treasury_df.dtypes

Record Date                             datetime64[us]
Classification ID                                int64
Classification Description                         str
Current Month Gross Receipts Amount            float64
Current Month Gross Outlay Amount              float64
Current Month Deficit Surplus Amount           float64
dtype: object

### 3. Currency Formatting

In the `treasury_df`, once again, becuase it is web data formatting is not applied. For the dollar amounts in this dataframe to be used for arithmitic, they must be in a numeric data type.

In [29]:
# Treasury dollar columns
treasury_dollars = [
    'Current Month Gross Receipts Amount',
    'Current Month Gross Outlay Amount',
    'Current Month Deficit Surplus Amount'
]

# Convert to float
treasury_df[treasury_dollars] = treasury_df[treasury_dollars].apply(pd.to_numeric, errors='coerce')

# Verify
print(treasury_df.dtypes)


Record Date                             datetime64[us]
Classification ID                                int64
Classification Description                         str
Current Month Gross Receipts Amount            float64
Current Month Gross Outlay Amount              float64
Current Month Deficit Surplus Amount           float64
dtype: object


### 4. Replace Headers

Shortening the column headers will make viewing a dataframe easier in a jupyter notebook cell.

In [30]:
treasury_df.columns

Index(['Record Date', 'Classification ID', 'Classification Description',
       'Current Month Gross Receipts Amount',
       'Current Month Gross Outlay Amount',
       'Current Month Deficit Surplus Amount'],
      dtype='str')

In [31]:
treasury_df.columns = ['Date', 'Class ID', 'Month', 'Gross Receipts', 'Gross Outlay Amount', 'Deficit Surplus Amount']
treasury_df.head()

,Date,Class ID,Month,Gross Receipts,Gross Outlay Amount,Deficit Surplus Amount
1,2025-12-31,57633931,October,3.267702e+11,5.842203e+11,2.574500e+11
2,2025-12-31,57633935,November,3.017543e+11,6.685171e+11,3.667628e+11
3,2025-12-31,57633940,December,4.544147e+11,5.411463e+11,8.673162e+10
4,2025-12-31,57633942,January,5.132943e+11,6.419345e+11,1.286402e+11
5,2025-12-31,57633946,February,2.964237e+11,6.034409e+11,3.070172e+11


In addition to abbreviating the existing headers, the field identifying month was named "Classification Description". This was likely an issue in the way the data was pulled from the HTML file. It was changed to `Month` to properly represent the field.

### 5. Remove Outliers

Outliers will be identified for the "Gross Receipts: fields using the IQR method.

In [35]:
# For each numeric column
Q1 = treasury_df['Gross Receipts'].quantile(0.25)
Q3 = treasury_df['Gross Receipts'].quantile(0.75)
IQR = Q3 - Q1

# Define outlier bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
treasury_df = treasury_df[
    (treasury_df['Gross Receipts'] >= lower_bound) & 
    (treasury_df['Gross Receipts'] <= upper_bound)
]

There were 6 outliers in the treasury data according to the boundry restrictions set forth in the prior cell.

In [36]:
treasury_df.shape

(76, 6)

### Ethical Implications

In this milestone, the HTML treasury data was extracted. Formatting the dates, filtering columns/values, currency formatting, replacing headers and removing outliers were among the transformations made to this raw source data.

The U.S Department of the Treasury offers digital file versions of the data which falls the use of it in line with Terms of Use. No additional known regulatory or legal guidelines were identified.

Risks handling this data include misrepresenting dollar amounts, by accident through formatting error or aggregation without consideration of outliers. Cleaning on this data is highly recommended before use in analysis and decision making.

An assumption was made in determining which values of Gross Receipts should be filtered out. Upper and lower tresholds were identified as limits to identify outliers.

This data was ethically sourced and verified, as it is from a credible government institution.

Any ethical implications identifed can be traced back to the source, identified, analyzed and answered for by refernecing the Privacy Policy and Terms of Use.

https://home.treasury.gov/subfooter/privacy-policy

https://www.treasurydirect.gov/legal-information/terms/